# Setup and Context

<img src="https://i.imgur.com/gugIA5r.png" width=700>

### Introduction

Dr Ignaz Semmelweis was a Hungarian physician born in 1818 who worked in the Vienna General Hospital. In the past people thought of illness as caused by "bad air" or evil spirits. But in the 1800s Doctors started looking more at anatomy, doing autopsies and started making arguments based on data. Dr Semmelweis suspected that something was going wrong with the procedures at Vienna General Hospital. Semmelweis wanted to figure out why so many women in maternity wards were dying from childbed fever (i.e., [puerperal fever](https://en.wikipedia.org/wiki/Postpartum_infections)).

<img src=https://i.imgur.com/lSyNUwR.png width=700>

Today you will become Dr Semmelweis. This is your office 👆. You will step into Dr Semmelweis' shoes and analyse the same data collected from 1841 to 1849.

### The Data Source

Dr Semmelweis published his research in 1861. I found the scanned pages of the [full text with the original tables in German](http://www.deutschestextarchiv.de/book/show/semmelweis_kindbettfieber_1861), but an excellent [English translation can be found here](http://graphics8.nytimes.com/images/blogs/freakonomics/pdf/the%20etiology,%20concept%20and%20prophylaxis%20of%20childbed%20fever.pdf).

<img src=https://i.imgur.com/6HfLtaC.png width=500>

### Upgrade plotly (only Google Colab Notebook)

Google Colab may not be running the latest version of plotly. If you're working in Google Colab, uncomment the line below, run the cell, and restart your notebook server. 

In [ ]:
# %pip install --upgrade plotly

### Import Statements

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

### Notebook Presentation

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

# Create locators for ticks on the time axis


from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

### Read the Data

In [ ]:
df_yearly = pd.read_csv('../data/annual_deaths_by_clinic.csv')
# parse_dates avoids DateTime conversion later
df_monthly = pd.read_csv('../data/monthly_deaths.csv', 
                      parse_dates=['date'])

In [ ]:
import pandas as pd

handwashing_start = pd.to_datetime('1846-06-01')


In [ ]:
# Step 1 — Create a Percentage Column

df_monthly['pct_deaths'] = df_monthly.deaths / df_monthly.births


In [ ]:
# Step 2 — Split the Data

before_washing = df_monthly[df_monthly.date < handwashing_start]
after_washing = df_monthly[df_monthly.date >= handwashing_start]

before_washing.head(1)

In [ ]:
after_washing.head(1)

In [ ]:
# Step 3 — Compute Average Death Rates

bw_rate = before_washing.deaths.sum() / before_washing.births.sum() * 100
aw_rate = after_washing.deaths.sum() / after_washing.births.sum() * 100

print(f'Average death rate BEFORE June 1846: {bw_rate:.4}%')
print(f'Average death rate AFTER June 1846: {aw_rate:.3}%')


### Use Box Plots to Show How the Death Rate Changed Before and After Handwashing

**Challenge**: 
* Use [NumPy's `.where()` function](https://numpy.org/doc/stable/reference/generated/numpy.where.html) to add a column to `df_monthly` that shows if a particular date was before or after the start of handwashing. 
* Then use plotly to create box plot of the data before and after handwashing. 
* How did key statistics like the mean, max, min, 1st and 3rd quartile changed as a result of the new policy?



In [ ]:
df_monthly['washing_hands'] = np.where(
    df_monthly.date < handwashing_start,
    'No',
    'Yes'
)


In [ ]:
box = px.box(
    df_monthly,
    x='washing_hands',
    y='pct_deaths',
    color='washing_hands',
    title='How Have the Stats Changed with Handwashing?'
)

box.update_layout(
    xaxis_title='Washing Hands?',
    yaxis_title='Percentage of Monthly Deaths'
)

box.show()


### Use Histograms to Visualise the Monthly Distribution of Outcomes

**Challenge**: Create a [plotly histogram](https://plotly.com/python/histograms/) to show the monthly percentage of deaths. 

* Use docs to check out the available parameters. Use the [`color` parameter](https://plotly.github.io/plotly.py-docs/generated/plotly.express.histogram.html) to display two overlapping histograms.
* The time period of handwashing is shorter than not handwashing. Change `histnorm` to `percent` to make the time periods comparable. 
* Make the histograms slighlty transparent
* Experiment with the number of bins on the histogram. Which number work well in communicating the range of outcomes?
* Just for fun, display your box plot on the top of the histogram using the `marginal` parameter. 

In [ ]:
hist = px.histogram(
    df_monthly,
    x='pct_deaths',
    color='washing_hands',
    nbins=30,
    opacity=0.6,
    barmode='overlay',
    histnorm='percent',
    marginal='box'
)

hist.update_layout(
    xaxis_title='Proportion of Monthly Deaths',
    yaxis_title='Percent'
)

hist.show()


### Use a Kernel Density Estimate (KDE) to visualise a smooth distribution

**Challenge**: Use [Seaborn's `.kdeplot()`](https://seaborn.pydata.org/generated/seaborn.kdeplot.html) to create two kernel density estimates of the `pct_deaths`, one for before handwashing and one for after. 

* Use the `shade` parameter to give your two distributions different colours. 
* What weakness in the chart do you see when you just use the default parameters?
* Use the `clip` parameter to address the problem. 


In [ ]:
plt.figure(dpi=200)
# By default the distribution estimate includes a negative death rate!
sns.kdeplot(before_washing.pct_deaths, fill=True)
sns.kdeplot(after_washing.pct_deaths, fill=True)
plt.title('Est. Distribution of Monthly Death Rate Before and After Handwashing')
plt.show()

In [ ]:
plt.figure(dpi=200)

sns.kdeplot(before_washing.pct_deaths, 
            fill=True,
            clip=(0,1),
            label='Before Handwashing')

sns.kdeplot(after_washing.pct_deaths, 
            fill=True,
            clip=(0,1),
            label='After Handwashing')

plt.title('Est. Distribution of Monthly Death Rate Before and After Handwashing')
plt.xlim(0, 0.40)

plt.legend()
plt.show()


### Use a T-Test to Show Statistical Significance

**Challenge**: Use a t-test to determine if the differences in the means are statistically significant or purely due to chance. 

If the p-value is less than 1% then we can be 99% certain that handwashing has made a difference to the average monthly death rate. 

* Import `stats` from scipy
* Use the [`.ttest_ind()` function](https://docs.scipy.org/]doc/scipy/reference/generated/scipy.stats.ttest_ind.html) to calculate the t-statistic and the p-value
* Is the difference in the average proportion of monthly deaths statistically significant at the 99% level? 



In [ ]:
import scipy.stats as stats

In [ ]:
t_stat, p_value = stats.ttest_ind(a=before_washing.pct_deaths, 
                                b=after_washing.pct_deaths)
print(f'p-palue is {p_value:.10f}')
print(f't-statstic is {t_stat:.4}')




## T TEST BREAKDOWN——Use a T-Test to Show Statistical Significance**

### Why Do We Need a Statistical Test?

We already saw that:

* Average death rate **before** handwashing ≈ 10.5%
* Average death rate **after** handwashing ≈ 5%

That looks like a huge drop.

But here’s the key question:

> Is this difference real… or could it just be random fluctuation?

A **t-test** helps us answer that.

---

## Understanding the Statistical Setup

### Step 1 — Define the Hypotheses

We formally test two competing ideas:

#### Null Hypothesis ( $H_0$ )

$[
H_0: \mu_{\text{before}} = \mu_{\text{after}}
]
$
There is **no real difference** in average monthly death rates.

Any difference we see is just random variation.

#### Alternative Hypothesis ( $H_1$ )

$[
H_1: \mu_{\text{before}} \neq \mu_{\text{after}}
]$

There *is* a real difference between the two averages.

---

## What the T-Test Actually Measures

The t-test compares:

$[
\text{Difference in means} \div \text{Expected random variation}
]$

In formula form:

$[
t =
\frac{\bar{x}_1 - \bar{x}_2}
{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}
]$

Where:

* $( \bar{x}_1 )$ = mean before handwashing
* $( \bar{x}_2 )$ = mean after handwashing
* $( s_1^2 ), ( s_2^2 )$ = sample variances
* $( n_1 ), ( n_2 ) $= number of months in each period

### Intuition

* Big numerator → large difference in means
* Small denominator → low variability
* Large t-value → difference unlikely to be random

---

## Running the Test in Python

```python
import scipy.stats as stats

t_stat, p_value = stats.ttest_ind(
    a=before_washing.pct_deaths,
    b=after_washing.pct_deaths
)

print(f'p-value is {p_value:.10f}')
print(f't-statistic is {t_stat:.4f}')
```

Your result:

* ( t = 3.804 )
* ( p = 0.0002504345 )

---

## What Is the P-Value?

The **p-value** answers this:

> If there were truly no real difference, how likely is it to observe a difference this extreme?

For a two-sided test:

$[
p =
2 \cdot P\left(T_{\nu} \ge |t_{\text{observed}}|\right)
]$

Equivalent CDF form:

$[
p =
2 \cdot \left(1 - F_{T_\nu}(|t_{\text{observed}}|)\right)
]$

Where:

* $( T_\nu )$ = t-distribution with $( \nu )$ degrees of freedom
* $( F_{T_\nu} )$ = cumulative distribution function

---

## Degrees of Freedom

If equal variances are assumed:

$[
\nu = n_1 + n_2 - 2
]$

If Welch’s version is used (`equal_var=False`), the degrees of freedom are approximated using:

$[
\nu \approx
\frac{\left(\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}\right)^2}
{\frac{\left(\frac{s_1^2}{n_1}\right)^2}{n_1-1} +
\frac{\left(\frac{s_2^2}{n_2}\right)^2}{n_2-1}}
]$

SciPy uses equal variance by default unless specified otherwise.

---

## Interpreting Our Result

We obtained:

$[
p = 0.0002504
]$

That means:

If handwashing had no real effect, there is only a **0.025% chance** we would observe a difference this large.

Since:

$[
p < 0.01
]$

We reject the null hypothesis at the **99% confidence level**.

---

## What Does This Mean in Plain English?

* The drop in death rate is **not random noise**.
* It is statistically significant.
* The data strongly supports the idea that handwashing reduced mortality.

---

## Why This Is So Powerful Historically

Semmelweis didn’t just observe a drop.

He could have said:

> The probability this happened by chance is 0.025%.

That is overwhelming statistical evidence.

---

## Beginner-Level Summary

### What We Did

1. Calculated monthly death rates.
2. Split data into before and after.
3. Compared averages.
4. Used a t-test to test if the difference is real.

### What the T-Test Does

* Measures how large the difference is relative to variation.
* Converts that into a probability (p-value).

### What the P-Value Means

* Very small p-value → very unlikely to be random.
* Our p-value is extremely small.
* Therefore: handwashing made a real difference.

---

If you'd like, next we can add a small visual intuition section explaining the t-distribution curve graphically so you fully *feel* what that p-value represents.


What do you conclude from your analysis, Doctor? 😊

<img src=https://i.imgur.com/rvjNVzQ.gif>